# Treinamento dos Modelos
## Nidus - Sistema de Análise de Comportamento Financeiro

Este notebook realiza a engenharia de atributos, treinamento e avaliação
dos modelos de classificação de transações e de perfil financeiro.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
import joblib
import warnings
warnings.filterwarnings('ignore')

## 1. Classificação de Transações

In [ ]:
df = pd.read_csv('../data/transacoes.csv')
X = df['descricao_normalizada']
y = df['categoria']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 3))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

modelo = LogisticRegression(max_iter=1000, multi_class='multinomial')
modelo.fit(X_train_vec, y_train)

In [ ]:
y_pred = modelo.predict(X_test_vec)
print(classification_report(y_test, y_pred))

## 2. Classificação de Perfil Financeiro

In [ ]:
df_perfil = pd.read_csv('../data/perfil.csv')

features = ['renda_mensal', 'nivel_endividamento', 'frequencia_poupanca',
            'proporcao_comprometimento_renda', 'proporcao_gastos_nao_essenciais']
X = df_perfil[features]
y = df_perfil['perfil_financeiro']

X = pd.get_dummies(X, columns=['frequencia_poupanca'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print(classification_report(y_test, y_pred))

## 3. Serialização dos Modelos

In [ ]:
joblib.dump({
    'modelo': modelo,
    'vectorizer': vectorizer
}, '../ml-service/models/modelo_transacoes.pkl')

joblib.dump(rf, '../ml-service/models/modelo_perfil.pkl')

print('Modelos serializados com sucesso!')

## 4. Conclusão

- Modelo de transações: F1 médio ponderado de X.XX (> 0.80 esperado)
- Modelo de perfil: F1 médio ponderado de X.XX (> 0.75 esperado)
- Modelos salvos em ml-service/models/